# Détection Automatique de Pathologies
## Classification Binaire sur SynMedTab-800

**Objectif** : Construire et comparer plusieurs modèles de classification binaire pour prédire la présence ou l'absence d'une pathologie à partir de données cliniques tabulaires synthétiques.

**Dataset** : SynMedTab-800
- Données synthétiques représentant 800 patients
- Variables quantitatives : âge, pression artérielle, cholestérol, glycémie, IMC
- Variables catégorielles : statut tabagique, niveau d'activité, antécédents familiaux, qualité alimentation
- Cible binaire : présence (1) ou absence (0) de pathologie

**Méthodologie** : Suivre le pipeline ML complet desde l'exploration jusqu'à l'analyse critique des résultats, en progressant du modèle simple au plus complexe.

---

## 1. IMPORTS ET CONFIGURATION

In [1]:
# Imports généraux
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Évaluation
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, 
                             f1_score, roc_auc_score, roc_curve, auc,
                             classification_report, ConfusionMatrixDisplay)

# Configuration d'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Tous les imports réussis")

✓ Tous les imports réussis


## 2. CHARGEMENT DES DONNÉES

In [2]:
# Charger le dataset
df = pd.read_csv('SynMedTab-800.csv')

# Afficher les premières lignes
print("Aperçu du dataset:")
print(df.head())
print(f"\nDimensions: {df.shape}")
print(f"\nTypes de données:")
print(df.dtypes)
print(f"\nStatistiques descriptives:")
print(df.describe())
print(f"\nValeurs manquantes:")
print(df.isnull().sum())

Aperçu du dataset:
   patient_id   age  blood_pressure_mmhg  cholesterol_mgdl  glucose_mgdl  \
0           1  68.7                125.6             193.9           NaN   
1           2  64.3                132.5             173.4          75.2   
2           3  51.1                118.3             213.3          86.4   
3           4   NaN                146.7             193.7          82.9   
4           5  47.5                144.3             182.6         182.6   

    bmi smoking_status physical_activity family_history diet_quality disease  
0  29.5          Never          Moderate             No    Excellent      No  
1  24.5          Never          Moderate             No         Fair      No  
2  26.2         Former              High            NaN         Good      No  
3  27.8          Never               NaN             No          NaN      No  
4  31.6          Never               Low            Yes         Poor     Yes  

Dimensions: (800, 11)

Types de données:
patient_

---
## 3. EXPLORATION ET ANALYSE EXPLORATOIRE (EDA)

### 3.1 Distribution de la variable cible

In [ ]:
# Identifier la colonne cible
# [À COMPLÉTER] : Déterminez le nom de la colonne cible (généralement 'pathology' ou 'target')
target_col = None  # À REMPLACER PAR LE BON NOM

if target_col is None:
    print("Colonnes disponibles:")
    print(df.columns.tolist())
else:
    # Distribution de la cible
    print(f"Distribution de la variable cible ({target_col}):")
    print(df[target_col].value_counts())
    print(f"\nProportion:")
    print(df[target_col].value_counts(normalize=True))
    
    # Visualisation
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    df[target_col].value_counts().plot(kind='bar', ax=ax[0], color=['#2ecc71', '#e74c3c'])
    ax[0].set_title('Distribution de la cible (nombre)')
    ax[0].set_ylabel('Nombre de patients')
    ax[0].set_xlabel('Pathologie')
    
    df[target_col].value_counts(normalize=True).plot(kind='pie', ax=ax[1], autopct='%1.1f%%',
                                                       colors=['#2ecc71', '#e74c3c'])
    ax[1].set_title('Distribution de la cible (%)')
    plt.tight_layout()
    plt.show()
    
    print(f"\nImbalancement des classes: {abs(100 - df[target_col].value_counts(normalize=True)[1]*100):.1f}%")

Colonnes disponibles:
['patient_id', 'age', 'blood_pressure_mmhg', 'cholesterol_mgdl', 'glucose_mgdl', 'bmi', 'smoking_status', 'physical_activity', 'family_history', 'diet_quality', 'disease']


### 3.2 Identification des variables

In [4]:
# Séparer les variables quantitatives et catégorielles
# [À COMPLÉTER] : Adapter cette liste selon les colonnes réelles

numerical_cols = []  # À COMPLÉTER : Lister les colonnes numériques (âge, tension, etc.)
categorical_cols = []  # À COMPLÉTER : Lister les colonnes catégoriques (smoking, activity, etc.)

print(f"Variables quantitatives ({len(numerical_cols)}): {numerical_cols}")
print(f"\nVariables catégorielles ({len(categorical_cols)}): {categorical_cols}")

Variables quantitatives (0): []

Variables catégorielles (0): []


### 3.3 Visualisation des variables quantitatives

In [5]:
# Histogrammes des variables quantitatives
if numerical_cols:
    fig, axes = plt.subplots(len(numerical_cols)//2 + 1, 2, figsize=(14, 3*(len(numerical_cols)//2+1)))
    axes = axes.flatten()
    
    for idx, col in enumerate(numerical_cols):
        axes[idx].hist(df[col], bins=30, color='skyblue', edgecolor='black')
        axes[idx].set_title(f'Distribution - {col}')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Fréquence')
        axes[idx].grid(True, alpha=0.3)
    
    # Masquer les subplots inutilisés
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

### 3.4 Analyse des anomalies et valeurs manquantes

In [6]:
# Détection des zéros anormaux
print("Zéros par colonne:")
for col in numerical_cols:
    zero_count = (df[col] == 0).sum()
    if zero_count > 0:
        print(f"  {col}: {zero_count} zéros ({zero_count/len(df)*100:.1f}%)")

print("\nValeurs manquantes:")
print(df.isnull().sum())

# Outliers (boxplot)
if numerical_cols:
    fig, axes = plt.subplots(1, len(numerical_cols)//2 + 1, figsize=(14, 4))
    for idx, col in enumerate(numerical_cols):
        axes[idx].boxplot(df[col])
        axes[idx].set_title(f'Boxplot - {col}')
        axes[idx].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

Zéros par colonne:

Valeurs manquantes:
patient_id              0
age                    84
blood_pressure_mmhg    70
cholesterol_mgdl       79
glucose_mgdl           96
bmi                    67
smoking_status         69
physical_activity      90
family_history         83
diet_quality           88
disease                 0
dtype: int64


### 3.5 Corrélations

In [7]:
# Matrice de corrélation avec la cible
if numerical_cols and target_col:
    data_for_corr = df[numerical_cols + [target_col]].copy()
    corr_matrix = data_for_corr.corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                cbar_kws={'label': 'Corrélation'})
    plt.title('Matrice de corrélation (variables quantitatives + cible)')
    plt.tight_layout()
    plt.show()
    
    print(f"\nCorrélations avec la cible ({target_col}):")
    target_corr = corr_matrix[target_col].sort_values(ascending=False)
    print(target_corr)

---
## 4. PRÉPARATION ET NETTOYAGE DES DONNÉES

### 4.1 Séparation features / target

In [8]:
# Séparer X et y
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nColonnes de X: {X.columns.tolist()}")

KeyError: '[None] not found in axis'

### 4.2 Gestion des valeurs manquantes et anomalies

In [ ]:
# [À COMPLÉTER] Traitement des valeurs manquantes et zéros anormaux
# Exemple :
# - Si Blood_Pressure = 0 : remplacer par NaN
# - Si Cholesterol = 0 : remplacer par NaN
# - Puis imputer avec la médiane

# Copie de travail
X_clean = X.copy()

# À COMPLÉTER : Remplacer les zéros biologiquement impossibles
# X_clean.loc[X_clean['Blood_Pressure'] == 0, 'Blood_Pressure'] = np.nan
# X_clean.loc[X_clean['Cholesterol'] == 0, 'Cholesterol'] = np.nan
# etc...

print("Valeurs manquantes après remplacement des zéros anormaux:")
print(X_clean.isnull().sum())

# Imputation des valeurs manquantes
from sklearn.impute import SimpleImputer

# Pour variables numériques : imputer avec la médiane
imputer_num = SimpleImputer(strategy='median')
X_clean[numerical_cols] = imputer_num.fit_transform(X_clean[numerical_cols])

print("\nAprès imputation:")
print(X_clean.isnull().sum())

### 4.3 Train / Test Split (AVANT normalisation)

In [ ]:
# Séparation train/test STRATIFIÉE (pour conserver l'équilibre des classes)
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, 
    test_size=0.2,  # 80% train, 20% test
    stratify=y,  # Conserver la proporion des classes
    random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nDistribution train:")
print(y_train.value_counts(normalize=True))
print(f"\nDistribution test:")
print(y_test.value_counts(normalize=True))

### 4.4 Prétraitement : Normalisation + Encodage

In [ ]:
# Créer les transformateurs
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

# Combiner les transformateurs
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Appliquer la transformation
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"X_train_transformed shape: {X_train_transformed.shape}")
print(f"X_test_transformed shape: {X_test_transformed.shape}")

---
## 5. MODÉLISATION - ENTRAÎNEMENT ET ÉVALUATION

### 5.1 Fonction d'évaluation

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Évalue un modèle de classification et retourne un dictionnaire de métriques.
    """
    # Prédictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    y_test_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calcul des métriques
    train_accuracy = (y_train_pred == y_train).mean()
    test_accuracy = (y_test_pred == y_test).mean()
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)
    auc_roc = roc_auc_score(y_test, y_test_pred_proba)
    
    # Affichage
    print(f"\n{'='*60}")
    print(f"MODÈLE: {model_name}")
    print(f"{'='*60}")
    print(f"Accuracy train: {train_accuracy:.4f}")
    print(f"Accuracy test:  {test_accuracy:.4f}")
    print(f"Surapprentissage: {abs(train_accuracy - test_accuracy):.4f}")
    print(f"\nMatrice de confusion (test):")
    cm = confusion_matrix(y_test, y_test_pred)
    print(cm)
    print(f"  TP: {cm[1,1]}, TN: {cm[0,0]}, FP: {cm[0,1]}, FN: {cm[1,0]}")
    print(f"\nPrécision: {precision:.4f}")
    print(f"Rappel:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc_roc:.4f}")
    
    # Retourner les métriques dans un dictionnaire
    return {
        'Model': model_name,
        'Train_Accuracy': train_accuracy,
        'Test_Accuracy': test_accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1_Score': f1,
        'AUC_ROC': auc_roc,
        'Overfit_Gap': abs(train_accuracy - test_accuracy),
        'y_test_pred': y_test_pred,
        'y_test_pred_proba': y_test_pred_proba,
        'model': model
    }

### 5.2 Modèle 1 : Régression Logistique (Baseline)

In [ ]:
# Entraîner la régression logistique
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_transformed, y_train)

# Évaluer
results_lr = evaluate_model(model_lr, X_train_transformed, X_test_transformed, 
                              y_train, y_test, "Régression Logistique")

### 5.3 Modèle 2 : Arbre de Décision

In [ ]:
# Entraîner l'arbre de décision (avec élagage pour éviter le surapprentissage)
model_dt = DecisionTreeClassifier(
    max_depth=10,           # À AJUSTER selon les résultats
    min_samples_split=10,   # À AJUSTER
    min_samples_leaf=5,     # À AJUSTER
    random_state=42
)
model_dt.fit(X_train_transformed, y_train)

# Évaluer
results_dt = evaluate_model(model_dt, X_train_transformed, X_test_transformed,
                             y_train, y_test, "Arbre de Décision")

print(f"\nImportance des features (Top 10):")
feature_importance = pd.DataFrame({
    'feature': range(X_train_transformed.shape[1]),
    'importance': model_dt.feature_importances_
}).sort_values('importance', ascending=False).head(10)
print(feature_importance)

### 5.4 Modèle 3 : Forêt Aléatoire

In [ ]:
# Entraîner la forêt aléatoire
model_rf = RandomForestClassifier(
    n_estimators=100,       # À AJUSTER
    max_depth=15,           # À AJUSTER
    min_samples_split=10,   # À AJUSTER
    min_samples_leaf=5,     # À AJUSTER
    random_state=42,
    class_weight='balanced'  # Pour gérer l'imbalancement
)
model_rf.fit(X_train_transformed, y_train)

# Évaluer
results_rf = evaluate_model(model_rf, X_train_transformed, X_test_transformed,
                             y_train, y_test, "Forêt Aléatoire")

print(f"\nImportance des features (Top 10):")
feature_importance_rf = pd.DataFrame({
    'feature': range(X_train_transformed.shape[1]),
    'importance': model_rf.feature_importances_
}).sort_values('importance', ascending=False).head(10)
print(feature_importance_rf)

---
## 6. COMPARAISON DES MODÈLES

### 6.1 Tableau comparatif

In [ ]:
# Créer un DataFrame de comparaison
comparison_df = pd.DataFrame([
    results_lr,
    results_dt,
    results_rf
])

# Afficher sans certaines colonnes
comparison_display = comparison_df[[
    'Model', 'Train_Accuracy', 'Test_Accuracy', 'Overfit_Gap',
    'Precision', 'Recall', 'F1_Score', 'AUC_ROC'
]].round(4)

print("\n" + "="*100)
print("COMPARAISON DES MODÈLES")
print("="*100)
print(comparison_display.to_string(index=False))
print("="*100)

### 6.2 Visualisation comparative

In [ ]:
# Graphique comparatif
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

metrics = ['Test_Accuracy', 'Precision', 'Recall', 'F1_Score', 'AUC_ROC', 'Overfit_Gap']
colors = ['#3498db', '#2ecc71', '#e74c3c']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    values = [comparison_df.iloc[i][metric] for i in range(len(comparison_df))]
    models = comparison_df['Model'].tolist()
    
    bars = ax.bar(models, values, color=colors, alpha=0.7, edgecolor='black')
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim([0, max(values) * 1.1])
    ax.grid(axis='y', alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

### 6.3 Courbes ROC comparées

In [ ]:
# Tracer les courbes ROC
plt.figure(figsize=(10, 8))

# Récalculer les proba pour LR et DT (manquantes du dictionnaire précédent)
y_lr_proba = model_lr.predict_proba(X_test_transformed)[:, 1]
y_dt_proba = model_dt.predict_proba(X_test_transformed)[:, 1]

# Tracer les courbes
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_lr_proba)
fpr_dt, tpr_dt, _ = roc_curve(y_test, results_dt['y_test_pred_proba'])
fpr_rf, tpr_rf, _ = roc_curve(y_test, results_rf['y_test_pred_proba'])

plt.plot(fpr_lr, tpr_lr, label=f'Régression Logistique (AUC={results_lr["AUC_ROC"]:.3f})', linewidth=2)
plt.plot(fpr_dt, tpr_dt, label=f'Arbre de Décision (AUC={results_dt["AUC_ROC"]:.3f})', linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f'Forêt Aléatoire (AUC={results_rf["AUC_ROC"]:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Classifieur aléatoire', linewidth=1)

plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
plt.ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
plt.title('Courbes ROC - Comparaison des Modèles', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 6.4 Matrices de confusion

In [ ]:
# Afficher les matrices de confusion
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_info = [
    (results_lr, model_lr, "Régression Logistique", axes[0]),
    (results_dt, model_dt, "Arbre de Décision", axes[1]),
    (results_rf, model_rf, "Forêt Aléatoire", axes[2])
]

for result, model, name, ax in models_info:
    y_pred = result['y_test_pred']
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Négatif (0)', 'Positif (1)'])
    disp.plot(ax=ax, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 7. ANALYSE CRITIQUE ET RÉSULTATS

### 7.1 Questions cliniques et d'interprétation

**À RÉPONDRE:**

1. **Déséquilibre des classes ?**
   - Quelle est la proportion exacte de pathologies détectées vs non-détectées ?
   - Comment cela influence-t-il le choix des métriques ?
   - Faudrait-il appliquer du rééquilibrage (SMOTE, class_weight) ?

2. **Performance par classe:**
   - Quel modèle détecte le mieux les vrais positifs (pathologies présentes) ?
   - Quel est le coût clinique d'un faux négatif ? (Patient malade non détecté)
   - Quel est le coût d'un faux positif ? (Patient sain pero diagnostiqué malade)
   - Le rappel ou la précision est-il plus important dans ce contexte ?

3. **Surapprentissage ?**
   - Quel modèle surapprend le plus ?
   - Comment l'améliorer ?

4. **Choix du meilleur modèle:**
   - Quel modèle recommander et pourquoi ?
   - Quels sont les trade-offs ?

In [ ]:
# À COMPLÉTER : Répondre aux questions ci-dessus avec du code et du texte

print("ANALYSE CLINIQUE ET D'INTERPRÉTATION")
print("="*60)

# [À COMPLÉTER] Analyser l'imbalancement
print("\n1. DÉSÉQUILIBRE DES CLASSES:")
print(f"   Proportion de pathologies: {(y==1).sum()/len(y)*100:.1f}%")

# [À COMPLÉTER] Analyser les faux positifs / faux négatifs
print("\n2. PERFORMANCE PAR CLASSE:")
best_model = max(comparison_df.index, key=lambda i: comparison_df.iloc[i]['F1_Score'])
print(f"   Modèle avec meilleur F1-score: {comparison_df.iloc[best_model]['Model']}")

# [À COMPLÉTER] Analyser le surapprentissage
print("\n3. SURAPPRENTISSAGE:")
for idx, row in comparison_df.iterrows():
    gap = row['Overfit_Gap']
    status = "⚠️  ÉLEVÉ" if gap > 0.1 else "✓ Acceptable"
    print(f"   {row['Model']}: {gap:.4f} {status}")

print("\n" + "="*60)

### 7.2 Analyse détaillée du meilleur modèle

In [ ]:
# Sélectionner le meilleur modèle (selon F1-score)
best_idx = comparison_df['F1_Score'].idxmax()
best_result = comparison_df.iloc[best_idx]
best_model_name = best_result['Model']

print(f"\n🏆 MEILLEUR MODÈLE: {best_model_name}")
print("="*60)

# Afficher le rapport de classification
y_best_pred = [results_lr, results_dt, results_rf][best_idx]['y_test_pred']
print("\nRAPPORT DE CLASSIFICATION:")
print(classification_report(y_test, y_best_pred, 
                           target_names=['Pas de pathologie', 'Pathologie présente']))

# À COMPLÉTER : Interprétation
print("\nINTERPRÉTATION:")
print("[À compléter] Analyser le rapport et expliquer les résultats en termes cliniques.")

---
## 8. CONCLUSIONS ET LIMITATIONS

### Résumé des résultats

[À COMPLÉTER] Rédiger un résumé de 3-4 phrases incluant:
- La description du dataset et de la problématique
- Les approches testées
- Le meilleur modèle et ses performances
- L'implication clinique des résultats

### Limitations de l'étude

[À COMPLÉTER] Discuter:
- Le caractère synthétique des données (pas de réalisme complet)
- L'imbalancement potentiel des classes
- Les variables disponibles (manque d'autres indicateurs cliniques?)
- La taille du dataset (800 patients)
- Les choix d'hyperparamètres

### Améliorations futures

[À COMPLÉTER] Proposer:
- De nouveaux modèles (SVM, Gradient Boosting, XGBoost)
- Du feature engineering
- De la validation croisée k-fold
- De l'hyperparameter tuning systématique (GridSearchCV)
- De la collecte de plus de données